# TinyML Fruits Pipeline (No Redownload) — Fruits-360 via KaggleHub or Local Cache

This notebook:
- **Does NOT re-download** the dataset if it's already present (local `/content` or Google Drive).
- Prefers **KaggleHub**: `kagglehub.dataset_download("moltean/fruits")`
- Falls back to an existing previously-downloaded folder if you already have one.
- Builds **TinyML-friendly** datasets:
  - **grayscale**
  - **small image size** (default: 32×32)
  - **subset of classes** (you choose)
  - labels remapped to **0..K-1**


In [ ]:
# =========================
# 0) Imports & Parameters
# =========================
import os, shutil, glob, difflib
from pathlib import Path
import tensorflow as tf

# ---- TinyML-friendly defaults ----
IMG_SIZE = (32, 32)      # 32x32 is far more feasible for tiny RAM than 96x96
BATCH_SIZE = 32
SEED = 123

# Set to True if you want to copy dataset into Google Drive for persistence
USE_GOOGLE_DRIVE = True

# Where to store dataset in Drive (if USE_GOOGLE_DRIVE = True)
DRIVE_DATASET_DIR = Path("/content/drive/MyDrive/datasets/moltean_fruits")

# If you previously downloaded/cloned a Fruits dataset, add hints here (optional)
PREVIOUS_LOCAL_DATASET_HINTS = [
    Path("/content/fruits-360-original-size"),
    Path("/content/fruits-360-original-size/fruits-360-original-size"),
    Path("/content/datasets/fruits-360-original-size"),
    Path("/content/datasets/moltean_fruits"),
]

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.19.0


In [ ]:
# =========================
# 1) (Optional) Mount Google Drive
# =========================
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Drive mounted")
else:
    print("ℹ️ Drive mount skipped")


Mounted at /content/drive
✅ Drive mounted


In [ ]:
# =========================
# 2) Locate dataset WITHOUT re-downloading
#    Priority:
#    A) Google Drive cache (if present)
#    B) KaggleHub cache (downloads once; future runs reuse)
#    C) Previously-downloaded local folders
# =========================

def find_split_root(start_dir: Path):
    """Find a dataset root that contains Training/Test[/Validation] (any casing)."""
    if start_dir is None or (not start_dir.exists()):
        return None

    def has_splits(p: Path):
        if not p.is_dir():
            return False
        names = {c.name.lower(): c for c in p.iterdir() if c.is_dir()}
        has_train = any(k in names for k in ["training", "train"])
        has_test  = any(k in names for k in ["test", "testing"])
        return has_train and has_test

    # quick checks
    if has_splits(start_dir):
        return start_dir

    # shallow search
    for p in start_dir.glob("*"):
        if has_splits(p):
            return p

    # recursive search (can be slower)
    for p in start_dir.rglob("*"):
        if has_splits(p):
            return p

    return None


def get_split_dirs(root: Path):
    """Return (TRAIN_DIR, VAL_DIR or None, TEST_DIR) from root."""
    names = {c.name.lower(): c for c in root.iterdir() if c.is_dir()}
    train = names.get("training") or names.get("train")
    test  = names.get("test") or names.get("testing")
    val   = names.get("validation") or names.get("valid") or names.get("val")
    if train is None or test is None:
        raise RuntimeError(f"Root {root} does not contain expected split folders.")
    return train, val, test


DATASET_ROOT = None

# A) If dataset already in Drive, use it
if USE_GOOGLE_DRIVE and DRIVE_DATASET_DIR.exists():
    candidate = find_split_root(DRIVE_DATASET_DIR)
    if candidate:
        DATASET_ROOT = candidate
        print(f"✅ Using dataset from Google Drive: {DATASET_ROOT}")

# B) KaggleHub (downloads once; later runs reuse cache)
if DATASET_ROOT is None:
    try:
        import kagglehub
        path = Path(kagglehub.dataset_download("moltean/fruits"))
        print("✅ KaggleHub dataset path:", path)
        candidate = find_split_root(path)
        if candidate:
            DATASET_ROOT = candidate
            print(f"✅ Using dataset from KaggleHub cache: {DATASET_ROOT}")
    except Exception as e:
        print("⚠️ KaggleHub failed or not available:", repr(e))

# C) Prior local hints
if DATASET_ROOT is None:
    for hint in PREVIOUS_LOCAL_DATASET_HINTS:
        candidate = find_split_root(hint)
        if candidate:
            DATASET_ROOT = candidate
            print(f"✅ Using previously-downloaded local dataset: {DATASET_ROOT}")
            break

if DATASET_ROOT is None:
    raise RuntimeError("Could not locate dataset. If KaggleHub failed, ensure it is available and try again, or add the correct folder to PREVIOUS_LOCAL_DATASET_HINTS.")

TRAIN_DIR, VAL_DIR, TEST_DIR = get_split_dirs(DATASET_ROOT)

print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR  :", VAL_DIR)
print("TEST_DIR :", TEST_DIR)


100%|██████████| 5.13G/5.13G [00:43<00:00, 127MB/s]

Extracting files...


✅ KaggleHub dataset path: /root/.cache/kagglehub/datasets/moltean/fruits/versions/78
✅ Using dataset from KaggleHub cache: /root/.cache/kagglehub/datasets/moltean/fruits/versions/78/fruits-360_original-size/fruits-360-original-size
TRAIN_DIR: /root/.cache/kagglehub/datasets/moltean/fruits/versions/78/fruits-360_original-size/fruits-360-original-size/Training
VAL_DIR  : /root/.cache/kagglehub/datasets/moltean/fruits/versions/78/fruits-360_original-size/fruits-360-original-size/Validation
TEST_DIR : /root/.cache/kagglehub/datasets/moltean/fruits/versions/78/fruits-360_original-size/fruits-360-original-size/Test


In [ ]:
# =========================
# 3) (Optional) Copy dataset to Google Drive for persistence
#    This avoids re-downloading in future sessions.
# =========================
if USE_GOOGLE_DRIVE:
    DRIVE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    drive_root = find_split_root(DRIVE_DATASET_DIR)
    if drive_root:
        print("✅ Drive already contains split folders. No copy needed:", drive_root)
    else:
        print("⏳ Copying dataset to Drive (one-time). This can take a while for large datasets...")
        target = DRIVE_DATASET_DIR / DATASET_ROOT.name
        if not target.exists():
            shutil.copytree(DATASET_ROOT, target)
            print("✅ Copied to:", target)
        else:
            print("✅ Target already exists. Skipping copy:", target)
else:
    print("ℹ️ Drive copy skipped (USE_GOOGLE_DRIVE=False)")


⏳ Copying dataset to Drive (one-time). This can take a while for large datasets...


KeyboardInterrupt: 

## Select your target classes (human-friendly names)

In [ ]:
# =========================
# 4) Pick classes (edit this list)
# =========================
target_classes = [
    "Apple Braeburn",
    "Banana",
    "Cherry 1",
    "Grape Blue",
    "Kiwi",
    "Lemon",
    "Orange",
    "Strawberry",
]
print("Requested classes:", target_classes)


Requested classes: ['Apple Braeburn', 'Banana', 'Cherry 1', 'Grape Blue', 'Kiwi', 'Lemon', 'Orange', 'Strawberry']


In [ ]:
# =========================
# 5) Resolve requested class names to actual folder names
# =========================
def list_class_folders(base_dir: Path):
    return sorted([d.name for d in base_dir.iterdir() if d.is_dir()])

def resolve_class_name(requested: str, available: list[str]):
    if requested in available:
        return requested
    pref = [a for a in available if a.lower().startswith(requested.lower())]
    if len(pref) == 1:
        return pref[0]
    close = difflib.get_close_matches(requested, available, n=1, cutoff=0.5)
    return close[0] if close else None

available_train = list_class_folders(TRAIN_DIR)
print("Train folder count:", len(available_train))
print("Example train folders:", available_train[:30])

mapping = {}
for c in target_classes:
    mapping[c] = resolve_class_name(c, available_train)

print("\nResolved mapping (requested -> actual folder):")
for k, v in mapping.items():
    print(f"  {k:15s} -> {v}")

subset_folders = [mapping[c] for c in target_classes if mapping.get(c) is not None]
if len(subset_folders) == 0:
    raise RuntimeError("None of the requested classes were found. Check TRAIN_DIR contents above.")

print("\n✅ Subset folder names used:", subset_folders)


Train folder count: 134
Example train folders: ['Almonds 1', 'Apple 10', 'Apple 11', 'Apple 12', 'Apple 13', 'Apple 14', 'Apple 17', 'Apple 18', 'Apple 19', 'Apple 5', 'Apple 7', 'Apple 8', 'Apple 9', 'Apple Core 1', 'Apple Red Yellow 2', 'Apple worm 1', 'Avocado Black 1', 'Avocado Black 2', 'Avocado Green 1', 'Banana 3', 'Banana 4', 'Beans 1', 'Blackberrie 1', 'Blackberrie 2', 'Blackberrie half rippen 1', 'Blackberrie not rippen 1', 'Blackberry 3', 'Cabbage red 1', 'Cactus fruit green 1', 'Cactus fruit red 1']

Resolved mapping (requested -> actual folder):
  Apple Braeburn  -> apple_braeburn_1
  Banana          -> Banana 4
  Cherry 1        -> Cherry 5
  Grape Blue      -> Grape not ripen 1
  Kiwi            -> None
  Lemon           -> None
  Orange          -> Pepper Orange 2
  Strawberry      -> Strawberry 3

✅ Subset folder names used: ['apple_braeburn_1', 'Banana 4', 'Cherry 5', 'Grape not ripen 1', 'Pepper Orange 2', 'Strawberry 3']


## Build TinyML-friendly datasets (grayscale + resize + remap labels)

In [ ]:
# =========================
# 6) Build datasets
#    - unbatched => correct filtering
#    - force grayscale
#    - resize to IMG_SIZE
#    - filter to subset folders
#    - remap labels 0..K-1
# =========================
def make_ds(dir_path: Path, subset_folder_names=None, shuffle=True):
    ds = tf.keras.utils.image_dataset_from_directory(
        str(dir_path),
        labels="inferred",
        label_mode="int",
        image_size=IMG_SIZE,
        batch_size=None,   # unbatched
        shuffle=shuffle,
        seed=SEED
    )
    class_names = ds.class_names

    def to_gray_and_resize(x, y):
        x = tf.image.rgb_to_grayscale(x)  # (H,W,1)
        x = tf.image.resize(x, IMG_SIZE)
        return x, y

    ds = ds.map(to_gray_and_resize, num_parallel_calls=tf.data.AUTOTUNE)

    if subset_folder_names is None:
        ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
        return ds, class_names

    new_order = [n for n in subset_folder_names if n in class_names]
    keep_indices = [class_names.index(n) for n in new_order]
    keep_tf = tf.constant(keep_indices, dtype=tf.int64)

    def keep_one(x, y):
        y = tf.cast(y, tf.int64)
        return tf.reduce_any(tf.equal(y, keep_tf))

    ds = ds.filter(keep_one)

    old_to_new = {class_names.index(name): i for i, name in enumerate(new_order)}
    lut_keys = tf.constant(list(old_to_new.keys()), dtype=tf.int64)
    lut_vals = tf.constant(list(old_to_new.values()), dtype=tf.int64)

    table = tf.lookup.StaticHashTable(
        tf.lookup.KeyValueTensorInitializer(lut_keys, lut_vals),
        default_value=-1
    )

    ds = ds.map(lambda x, y: (x, table.lookup(tf.cast(y, tf.int64))),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.filter(lambda x, y: tf.greater_equal(y, 0))

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, new_order

train_ds, subset_class_names = make_ds(TRAIN_DIR, subset_folders, shuffle=True)

if VAL_DIR is not None:
    val_ds, _ = make_ds(VAL_DIR, subset_folders, shuffle=False)
else:
    print("⚠️ No Validation folder found. Using Test as validation.")
    val_ds, _ = make_ds(TEST_DIR, subset_folders, shuffle=False)

test_ds, _ = make_ds(TEST_DIR, subset_folders, shuffle=False)

print("✅ Final subset classes (order):", subset_class_names)
print("K =", len(subset_class_names))

for xb, yb in train_ds.take(1):
    print("Train batch X shape:", xb.shape)  # should be (B, IMG_SIZE[0], IMG_SIZE[1], 1)
    print("Train batch Y shape:", yb.shape)


Found 46761 files belonging to 134 classes.
Found 23387 files belonging to 134 classes.
Found 23244 files belonging to 134 classes.
✅ Final subset classes (order): ['apple_braeburn_1', 'Banana 4', 'Cherry 5', 'Grape not ripen 1', 'Pepper Orange 2', 'Strawberry 3']
K = 6
Train batch X shape: (32, 32, 32, 1)
Train batch Y shape: (32,)


## Ultra-small CNN (TinyML-friendly)

In [ ]:
# =========================
# 7) Ultra-small Tiny CNN
# =========================
from tensorflow import keras
from tensorflow.keras import layers

num_classes = len(subset_class_names)

model = keras.Sequential([
    layers.Input(shape=IMG_SIZE + (1,)),
    layers.Rescaling(1./255),

    layers.SeparableConv2D(6, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.SeparableConv2D(12, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.SeparableConv2D(16, 3, padding="same", activation="relu"),
    layers.GlobalAveragePooling2D(),

    layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()
print("Model expects input:", model.input_shape)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_2 (Rescaling)         │ (None, 32, 32, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_6              │ (None, 32, 32, 6)      │            21 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 16, 16, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_7              │ (None, 16, 16, 12)     │           138 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 8, 8, 12)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_8              │ (None, 8, 8, 16)       │           316 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 16)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6)              │           102 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 577 (2.25 KB)

 Trainable params: 577 (2.25 KB)

 Non-trainable params: 0 (0.00 B)

Model expects input: (None, 32, 32, 1)


## Train with early stopping and evaluate

In [ ]:
# =========================
# 8) Training
# =========================
from tensorflow import keras

EPOCHS = 6  # upper bound; early stopping stops earlier

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"✅ Test accuracy: {test_acc:.4f}")


Epoch 1/6
69/69 ━━━━━━━━━━━━━━━━━━━━ 97s 1s/step - accuracy: 0.1958 - loss: 1.7870 - val_accuracy: 0.2078 - val_loss: 1.7710
Epoch 2/6
69/69 ━━━━━━━━━━━━━━━━━━━━ 96s 1s/step - accuracy: 0.2105 - loss: 1.7630 - val_accuracy: 0.2078 - val_loss: 1.7323
Epoch 3/6
69/69 ━━━━━━━━━━━━━━━━━━━━ 135s 2s/step - accuracy: 0.2321 - loss: 1.7014 - val_accuracy: 0.3473 - val_loss: 1.5529
Epoch 4/6
69/69 ━━━━━━━━━━━━━━━━━━━━ 95s 1s/step - accuracy: 0.4300 - loss: 1.4409 - val_accuracy: 0.5706 - val_loss: 1.1263
Epoch 5/6
69/69 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.6300 - loss: 0.9915 - val_accuracy: 0.7803 - val_loss: 0.7048
Epoch 6/6
69/69 ━━━━━━━━━━━━━━━━━━━━ 104s 2s/step - accuracy: 0.8284 - loss: 0.6247 - val_accuracy: 0.8897 - val_loss: 0.4714
✅ Test accuracy: 0.8907


## Export INT8 TFLite model

In [ ]:
# =========================
# 9) INT8 Quantization to TFLite (for microcontrollers)
# =========================
import numpy as np

def representative_data_gen():
    for images, _ in train_ds.take(50):
        yield [tf.cast(images, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

OUT_TFLITE = "tiny_fruits_int8.tflite"
with open(OUT_TFLITE, "wb") as f:
    f.write(tflite_model)

print("✅ Saved:", OUT_TFLITE, "| Size (bytes):", os.path.getsize(OUT_TFLITE))


Saved artifact at '/tmp/tmpudph324b'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 1), dtype=tf.float32, name='keras_tensor_18')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  133880236611664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133880236612624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133880236611280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133880236613200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133880236610896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133880236610704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133880236613392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133880236612432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133880236612240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133880236613584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1338802366137

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


✅ Saved: tiny_fruits_int8.tflite | Size (bytes): 7752


## Save outputs (TFLite + labels) to Google Drive

In [ ]:
# =========================
# 10) Save outputs to Drive (optional)
# =========================
if USE_GOOGLE_DRIVE:
    out_dir = Path("/content/drive/MyDrive/tinyml_fruits_outputs")
    out_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy2(OUT_TFLITE, out_dir / OUT_TFLITE)
    print("✅ Copied TFLite to:", out_dir / OUT_TFLITE)

    labels_path = out_dir / "labels.txt"
    with open(labels_path, "w") as f:
        for name in subset_class_names:
            f.write(name + "\n")
    print("✅ Saved labels to:", labels_path)
else:
    print("ℹ️ Drive output skipped")


✅ Copied TFLite to: /content/drive/MyDrive/tinyml_fruits_outputs/tiny_fruits_int8.tflite
✅ Saved labels to: /content/drive/MyDrive/tinyml_fruits_outputs/labels.txt


In [ ]:
interpreter = tf.lite.Interpreter(model_path="tiny_fruits_int8.tflite")
interpreter.allocate_tensors()

total = 0
for t in interpreter.get_tensor_details():
    shape = t['shape']
    dtype = t['dtype']
    size = tf.dtypes.as_dtype(dtype).size
    total += size * tf.reduce_prod(shape)

print("Approx tensor bytes:", int(total))


Approx tensor bytes: 17705


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path="tiny_fruits_int8.tflite")
interpreter.allocate_tensors()
print("Input:", interpreter.get_input_details()[0])
print("Output:", interpreter.get_output_details()[0])


Input: {'name': 'serving_default_keras_tensor_18:0', 'index': 0, 'shape': array([ 1, 32, 32,  1], dtype=int32), 'shape_signature': array([-1, 32, 32,  1], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.9999000430107117, -128), 'quantization_parameters': {'scales': array([0.99990004], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}
Output: {'name': 'StatefulPartitionedCall_1:0', 'index': 26, 'shape': array([1, 6], dtype=int32), 'shape_signature': array([-1,  6], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.00390625, -128), 'quantization_parameters': {'scales': array([0.00390625], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}


In [ ]:
# run locally or in colab
data = open("tiny_fruits_int8.tflite","rb").read()
with open("model_data.h","w") as f:
    f.write("const unsigned char model_tflite[] = {")
    f.write(",".join(str(b) for b in data))
    f.write("};\n")
    f.write(f"const unsigned int model_tflite_len = {len(data)};\n")
